# Getting Started

This guide introduces the core concepts of dags and shows you how to get started.

## Core Concept

dags works by analyzing function signatures to build a dependency graph. If a function
has a parameter named `x`, and another function is named `x`, then the first function
depends on the second.

In [ ]:
def x():
    return 1

In [ ]:
def y(x):
    return x + 1

`y` depends on `x` because its parameter is named `x`.

## Basic Usage

### Creating a Combined Function

We start by defining individual functions that represent parts of a simple tax model. In a real project, each function might live in its own module — what matters is that dags can discover dependencies from parameter names alone.

In [ ]:
import dags


def income():
    return 50000


def tax_rate():
    return 0.3


def tax(income, tax_rate):
    return income * tax_rate


def net_income(income, tax):
    return income - tax


all_functions = {
    "income": income,
    "tax_rate": tax_rate,
    "tax": tax,
    "net_income": net_income,
}

We have collected all four functions into a single dictionary. This is the typical starting point: in a real economic model you might have dozens of functions computing income, taxes, transfers, utility, etc.

Now we use `concatenate_functions` to combine them into **one function** that we can call. We pass:
- `functions` — the dictionary of all available functions
- `targets` — which outputs we actually want (here: `net_income` and `tax`)
- `return_type="dict"` — return results as a dictionary (the default is a tuple)

Since every input (`income`, `tax_rate`) is already provided by a function in the dictionary, the combined function needs no arguments at all:

In [ ]:
combined_no_inputs = dags.concatenate_functions(
    functions=all_functions,
    targets=["net_income", "tax"],
    return_type="dict",
)

In [ ]:
combined_no_inputs()

### Providing External Inputs

In the example above, `income` and `tax_rate` were hard-wired as functions that always return the same value. That is rarely useful in practice — typically these values come from data or vary across scenarios.

A more realistic setup is to include **only the functions that actually compute something**, and leave the raw inputs to be provided when calling the combined function:

In [ ]:
tax_and_net_income = {"tax": tax, "net_income": net_income}

When we combine just these two functions, dags notices that `income` and `tax_rate` are not produced by any function in the dictionary. It turns them into **required arguments** of the combined function.

In [ ]:
combined_with_inputs = dags.concatenate_functions(
    functions=tax_and_net_income,
    targets=["net_income", "tax"],
    return_type="dict",
)

Calling without arguments now raises an error, because dags needs the external inputs:

In [ ]:
combined_with_inputs()

We provide the missing inputs as keyword arguments:

In [ ]:
combined_with_inputs(income=50000, tax_rate=0.3)

### Return Types

By default, `concatenate_functions` returns results as a tuple (in the order of `targets`). We used `return_type="dict"` above to get a labeled dictionary instead, which is usually more convenient:

In [ ]:
combined_tuple = dags.concatenate_functions(
    functions=all_functions,
    targets=["net_income", "tax"],
)

combined_tuple()

## Inspecting the DAG

Under the hood, dags builds a dependency graph to figure out which functions to run and in what order. You can inspect this graph directly using `create_dag`, which returns a [networkx](https://networkx.org/) `DiGraph`:

In [ ]:
dag = dags.create_dag(
    functions=all_functions,
    targets=["net_income"],
)

In [ ]:
list(dag.nodes())

In [ ]:
list(dag.edges())

## Finding Dependencies

`get_ancestors` is useful for understanding which functions (and inputs) contribute to a given target. This can help you debug unexpected results or understand a large model's structure:

In [ ]:
dags.get_ancestors(
    functions=all_functions,
    targets=["net_income"],
)

## Inspecting Functions

dags provides utility functions for working with function signatures. These are handy when you need to programmatically inspect or modify functions.

### Getting Function Arguments

`get_free_arguments` returns the parameter names of a function (excluding those with defaults):

In [ ]:
def my_func(a, b, c=1):
    return a + b + c


dags.get_free_arguments(my_func)

### Getting Type Annotations

`get_annotations` returns type annotations:

In [ ]:
def my_func(a: int, b: float) -> float:
    return a + b


dags.get_annotations(my_func)

## Renaming Arguments

When integrating functions from different sources — say, a tax module that uses `gross_income` and a benefits module that uses `total_income` for the same quantity — you need to align parameter names so dags can connect them. `rename_arguments` creates a wrapper with renamed parameters:

In [ ]:
def original(x, y):
    return x + y


renamed = dags.rename_arguments(original, mapper={"x": "a", "y": "b"})

dags.get_free_arguments(renamed)

## Next Steps

- Learn about [common usage patterns](usage_patterns.ipynb) from real projects
- Explore [tree structures](tree.ipynb) for nested data